In [1]:
import pandas as pd
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import StackingRegressor

from scipy.stats import uniform, randint
from sklearn.metrics import r2_score

In [ ]:
dev = pd.read_csv("../data/processed/dev_data.csv")
test = pd.read_csv("../data/processed/test_data.csv")

In [3]:
dev

,time,magnitude,place,depth,latitude,longitude,city,province,direction,km,...,latitude_rad,longitude_rad,lat_cos,lat_sin,lon_cos,lon_sin,mag_mean_30d,mag_trend_30d,activity_ratio,last_event_km
0,1985-01-02 22:57:43.090,2.6,6 km S Civitella Alfedena (AQ),5.8,41.7150,13.9410,Civitella Alfedena,AQ,S,6.0,...,0.728064,0.243316,0.746464,0.665426,0.970544,0.240923,1.244118,-0.011108,0.411765,5.0
1,1985-01-03 00:46:16.990,2.5,6 km NE Picinisco (FR),6.1,41.6770,13.9310,Picinisco,FR,NE,6.0,...,0.727401,0.243142,0.746905,0.664931,0.970586,0.240753,1.200000,-0.100000,0.333333,7.0
2,1985-01-05 08:32:41.900,2.6,3 km NW Fiastra (MC),2.8,43.0560,13.1390,Fiastra,MC,NW,3.0,...,0.751469,0.229319,0.730687,0.682713,0.973821,0.227314,1.075909,-0.000322,0.331818,5.0
3,1985-01-05 10:13:26.800,2.2,1 km E San Raffaele Cimena (TO),20.8,45.1500,7.8600,San Raffaele Cimena,TO,E,1.0,...,0.788016,0.137183,0.705253,0.708956,0.990605,0.136753,0.800000,0.000000,0.000000,3.0
4,1985-01-06 03:52:23.380,2.4,3 km W Gubbio (PG),10.0,43.3570,12.5370,Gubbio,PG,W,3.0,...,0.756722,0.218812,0.727090,0.686542,0.976156,0.217070,1.020492,-0.000914,0.286885,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334276,2020-11-04 06:04:32.480,1.5,5 km E Santo Stefano in Aspromonte (RC),8.8,38.1870,15.8427,Santo Stefano in Aspromonte,RC,E,5.0,...,0.666489,0.276507,0.785997,0.618230,0.962015,0.272997,1.459091,0.002190,0.212121,5.0
334277,2020-11-04 06:05:06.920,1.2,5 km NW Vernio (PO),9.8,44.0908,11.1255,Vernio,PO,NW,5.0,...,0.769530,0.194177,0.718238,0.695797,0.981207,0.192959,1.233333,-0.250000,0.666667,2.0
334278,2020-11-04 06:15:11.050,1.0,2 km NW Vernio (PO),9.5,44.0743,11.1467,Vernio,PO,NW,2.0,...,0.769242,0.194547,0.718438,0.695591,0.981135,0.193322,1.233333,-0.250000,0.666667,2.0
334279,2020-11-04 06:27:45.910,0.5,5 km W Caldarola (MC),14.2,43.1322,13.1643,Caldarola,MC,W,5.0,...,0.752799,0.229760,0.729778,0.683684,0.973721,0.227744,1.075909,-0.000322,0.331818,5.0


In [4]:
dev = dev[[
    "province",
    "magnitude", 
    "depth", 
    "depth_category",
    "latitude", 
    "longitude",
    "lat_sin",
    "lat_cos",
    "lon_sin",
    "lon_cos",
    "km",
    "km_squared",
    "direction_angle_sin",
    "direction_angle_cos",
    "is_night", 
    "season", 
    "month_sin", 
    "month_cos", 
    "hour_sin", 
    "hour_cos",
    "mag_mean_30d",
    "mag_trend_30d",
    "activity_ratio",
    "last_event_km"
]]

X_dev = dev.drop(columns=["magnitude"])
y_dev = dev["magnitude"]

In [5]:
test = test[[
    "province",
    "magnitude", 
    "depth", 
    "depth_category",
    "latitude", 
    "longitude",
    "lat_sin",
    "lat_cos",
    "lon_sin",
    "lon_cos",
    "km",
    "km_squared",
    "direction_angle_sin",
    "direction_angle_cos",
    "is_night", 
    "season", 
    "month_sin", 
    "month_cos", 
    "hour_sin", 
    "hour_cos",
    "mag_mean_30d",
    "mag_trend_30d",
    "activity_ratio",
    "last_event_km"
]]

X_test = test.drop(columns=["magnitude"])
y_test = test["magnitude"]

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.1765,
    random_state=42
)

In [7]:
numeric_features = [
    "latitude", "longitude", "depth", "km", "is_night",
    "lat_cos", "lat_sin", "lon_cos", "lon_sin", "km_squared",
    "mag_mean_30d", "mag_trend_30d", "activity_ratio", "last_event_km"
]

categorical_features = ["province"]
onehot_features = ["season", "depth_category"]

cyclic_features = [
    "direction_angle_sin",
    "direction_angle_cos",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos"
]

In [ ]:
# Definizione del preprocessor che applica trasformazioni diverse a gruppi di colonne
preprocessor = ColumnTransformer(
    transformers=[
        # Applica StandardScaler alle feature numeriche
        ("num", StandardScaler(), numeric_features),
        
        # Applica TargetEncoder alle feature categoriche, utile per modelli non lineari
        ("cat", TargetEncoder(smoothing=10), categorical_features),
        
        # Applica OneHotEncoder alle feature categoriche nominali, gestendo i valori sconosciuti
        ("onehot", OneHotEncoder(handle_unknown="ignore"), onehot_features),
        
        # Lascia passare le feature cicliche (es. giorno della settimana, mese) così come sono
        ("cyclic", "passthrough", cyclic_features)
    ],
    
    # Tutte le feature non specificate vengono mantenute invariate
    remainder="passthrough"
)

# Applica il preprocessor ai dati di training (fit + transform)
X_train_tr = preprocessor.fit_transform(X_train, y_train)

# Salva il preprocessor su disco per usi futuri (es. test, deployment)
joblib.dump(preprocessor, "../encoder/preprocessor.pkl")

# Applica il preprocessor già addestrato ai dati di validazione
X_val_tr = preprocessor.transform(X_val)

# Applica il preprocessor anche ai dati di test
X_test_tr = preprocessor.transform(X_test)


In [ ]:
# Dizionario dei parametri iperottimizzabili per XGBoost
param_dist_xgb = {
    "learning_rate": [0.01, 0.2, 0.03, 0.4, 0.05],        # Tasso di apprendimento
    "max_depth": [4, 8, 10],                             # Profondità massima degli alberi
    "min_child_weight": [5, 7, 10],                      # Peso minimo richiesto per un nodo foglia
    "subsample": [0.5, 0.8, 1.0],                        # Percentuale di righe da campionare
    "colsample_bytree": [0.3, 0.6, 1.0],                 # Percentuale di colonne da usare per ogni albero
    "gamma": [0.1, 0.5],                                 # Guadagno minimo per dividere un nodo
    "reg_alpha": [0, 0.1, 0.5],                          # Regolarizzazione L1
    "reg_lambda": [0.5, 2.5, 5.0],                       # Regolarizzazione L2
    "max_bin": [128, 512]                                # Numero massimo di bin per il discretizzatore
}

# Definizione del modello base XGBoost per regressione
base_model = XGBRegressor(
    objective="reg:squarederror",       # Funzione obiettivo per regressione
    n_estimators=1000,                  # Numero massimo di alberi
    enable_categorical=True,            # Abilita supporto per variabili categoriche (se presenti)
    eval_metric="rmse"                  # Metriche di valutazione durante il training
)

# Creazione di un oggetto K-Fold Cross-Validation con 5 suddivisioni
CV = KFold(n_splits=5, shuffle=True, random_state=42)

# Ricerca casuale di iperparametri con cross-validation
random_search = RandomizedSearchCV(
    estimator=base_model,                        # Modello di base da ottimizzare
    param_distributions=param_dist_xgb,          # Spazio dei parametri
    n_iter=50,                                   # Numero di combinazioni da testare
    scoring="neg_root_mean_squared_error",       # Metica da massimizzare (valori negativi)
    cv=CV,                                       # Strategia di validazione incrociata
    verbose=0,                                   # Nessun output durante la ricerca
    n_jobs=-1,                                   # Usa tutti i core disponibili
    random_state=42                              # Per riproducibilità
)

# Parametri aggiuntivi da passare al metodo fit (per early stopping e validazione)
fit_params = {
    "eval_set": [(X_val_tr, y_val)],  # Set di validazione per monitoraggio del training
    "verbose": False                  # Disattiva output dettagliato del training
}

# Avvio della ricerca degli iperparametri migliori
random_search.fit(
    X_train_tr,
    y_train,
    **fit_params
)

# Recupero del punteggio RMSE migliore ottenuto (in positivo)
best_rmse = -random_search.best_score_

# Recupero dei parametri che hanno portato al miglior modello
best_params_random = random_search.best_params_

# Stampa del miglior punteggio e dei parametri ottimizzati
print(f"Best RMSE: {best_rmse:.4f}")
print(f"Best params: {best_params_random}")

# Estrazione del miglior modello ottenuto dalla RandomizedSearchCV
best_model = random_search.best_estimator_

# Predizione sul set di validazione usando il miglior modello
y_val_pred = best_model.predict(X_val_tr)

# Calcolo del coefficiente di determinazione R² sulla validazione
val_r2 = r2_score(y_val, y_val_pred)

# Stampa del punteggio R²
print(f"Validation R²: {val_r2:.4f}")


In [ ]:
# Definizione del modello XGBoost per regressione
xgb = XGBRegressor(
    objective="reg:squarederror",   # Funzione obiettivo per regressione (MSE)
    n_estimators=1000,              # Numero massimo di alberi
    enable_categorical=True,        # Abilita supporto per variabili categoriche (se presenti)
    eval_metric="rmse"              # Valutazione del modello tramite RMSE
)

# Griglia di iperparametri da testare con ricerca esaustiva (GridSearch)
param_grid_xgb = {
    "learning_rate": [0.025, 0.03, 0.035],      # Tasso di apprendimento (basso per apprendimento lento e accurato)
    "max_depth": [10],                         # Profondità fissa degli alberi
    "min_child_weight": [6, 7, 8],             # Peso minimo per foglia (regolarizzazione strutturale)
    "subsample": [0.7, 0.8, 0.9],              # Percentuale di campionamento dei dati
    "colsample_bytree": [0.5, 0.6, 0.7],       # Percentuale di colonne per ogni albero
    "gamma": [0.05, 0.1, 0.15],                # Guadagno minimo per split (riduce overfitting)
    "reg_alpha": [0.5],                        # Regolarizzazione L1 (lasso)
    "reg_lambda": [0.5],                       # Regolarizzazione L2 (ridge)
    "max_bin": [512]                           # Numero massimo di bin (per dati continui o ordinati)
}

# K-Fold cross-validation con 5 fold e shuffle dei dati
CV = KFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV: ricerca esaustiva su tutti i parametri specificati nella griglia
grid = GridSearchCV(
    estimator=xgb,                               # Modello da ottimizzare
    param_grid=param_grid_xgb,                   # Spazio iperparametrico da esplorare
    scoring="neg_root_mean_squared_error",       # Metica da massimizzare (valore negativo di RMSE)
    cv=CV,                                       # Strategia di cross-validation
    verbose=0,                                   # Nessun log durante la ricerca
    n_jobs=-1                                    # Usa tutti i core disponibili per la ricerca
)

# Parametri da passare al metodo `fit`, inclusi dati di validazione per monitoraggio
fit_params = {
    "eval_set": [(X_val_tr, y_val)],  # Dati di validazione per early stopping o controllo del learning
    "verbose": False                  # Disattiva output dettagliato durante il training
}

# Esegue la grid search e addestra il modello su ciascuna combinazione di parametri
grid.fit(
    X_train_tr,
    y_train,
    **fit_params
)

# Recupera il miglior punteggio (in positivo) e la combinazione migliore di parametri
best_rmse = -grid.best_score_
best_params = grid.best_params_

# Stampa del miglior RMSE e dei corrispondenti iperparametri trovati
print(f"Best RMSE: {best_rmse:.4f}")
print(f"Best params: {best_params}")

# Estrazione del miglior modello trovato tramite GridSearchCV
best_xgb = grid.best_estimator_

# Predizione sul set di test usando il miglior modello
y_pred_xgb = best_xgb.predict(X_test_tr)

# Calcolo del coefficiente di determinazione R² sul test set
r2 = r2_score(y_test, y_pred_xgb)
print(f"R²: {r2:.4f}")


In [11]:
best_params =  {'subsample': 0.8, 'reg_lambda': 5.0, 'reg_alpha': 0.5, 'min_child_weight': 7, 'max_depth': 10, 'max_bin': 512, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.6}

In [ ]:
# Inizializzazione del modello XGBoost con i migliori iperparametri trovati
xgb_model = XGBRegressor(
    **best_params,                       # Inserisce automaticamente i migliori iperparametri dalla grid/random search
    objective="reg:squarederror",       # Obiettivo di regressione (errore quadratico medio)
    n_estimators=1000,                  # Numero massimo di alberi
    enable_categorical=True,            # Abilita supporto per feature categoriche
    eval_metric="rmse"                  # Usa l'RMSE come metrica di valutazione
)

# Parametri per il metodo fit: include un set di validazione per monitorare le prestazioni
fit_params = {
    "eval_set": [(X_val_tr, y_val)],    # Dati di validazione per il monitoraggio dell'apprendimento
    "verbose": False                    # Disattiva output dettagliato durante l'addestramento
}

# Addestramento del modello sui dati di training trasformati
xgb_model.fit(
    X_train_tr,
    y_train,
    **fit_params                        # Applica i parametri di fit, incluso il validation set
)

# Predizione sul set di test
y_pred_xgb = xgb_model.predict(X_test_tr)

# Calcolo del coefficiente di determinazione R² (valuta la bontà del modello)
r2 = r2_score(y_test, y_pred_xgb)


In [ ]:
# Salvataggio del modello XGBoost addestrato su file .pkl per usi futuri o produzione
joblib.dump(xgb_model, "../model/xgb_magnitude_model.pkl")

['models/xgb_magnitude_model.pkl']